# Applied AI Assignment 1: Multi-Domain Sentiment Analysis

This notebook uses dataset option 1 from the assignment brief. Run every cell from top to bottom after installing requirements. The notebook is intentionally unexecuted in this repository because the official JHU archive could not be downloaded in the build environment.

In [ ]:
from pathlib import Path
import sys, subprocess
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from assignment1.data import load_multidomain_dataset, dataset_summary
from assignment1.experiment import run_experiment
DATA_DIR = ROOT / 'data' / 'raw' / 'processed_acl'
RESULTS_DIR = ROOT / 'results'
print(ROOT)

## 1. Download the official dataset

The selected archive is `processed_acl.tar.gz` (about 19 MB). Raw data are not committed to Git.

In [ ]:
if not DATA_DIR.exists() or not any(DATA_DIR.rglob('positive.review')):
    subprocess.run([sys.executable, str(ROOT / 'scripts' / 'download_dataset.py'), '--variant', 'processed_acl'], check=True)
else:
    print('Dataset folder already exists.')

In [ ]:
dataset = load_multidomain_dataset(DATA_DIR)
summary = pd.DataFrame(dataset_summary(dataset))
display(summary)
ax = summary.set_index('domain')[['positive', 'negative']].plot(kind='bar', figsize=(9,4))
ax.set_title('Label counts by product domain')
plt.tight_layout()
plt.show()

## 2. Train and evaluate the models

Default source domain: Books. Target domains: DVD, Electronics and Kitchen when available. The models are Naive Bayes, Logistic Regression, Linear SVM, DNN and Residual MLP.

In [ ]:
source = 'books'
targets = [d for d in ['dvd', 'electronics', 'kitchen'] if d in dataset and d != source]
metrics = run_experiment(DATA_DIR, RESULTS_DIR, source_domain=source, target_domains=targets, max_features=10000, epochs=20, seed=42)
display(metrics.sort_values(['evaluation', 'f1'], ascending=[True, False]))

## 3. Interpret the results

Use the generated CSV, confusion matrices and F1-score plot. In your own words, identify the best in-domain model, the model that transfers best across domains, the largest performance drop and possible reasons for that drop. Recompile `report/main.tex` after the experiment. Do not submit this notebook without visible outputs from a real run.